# 00 — Download & Build Training Dataset

Downloads a small subset of **cpg0016-jump** (source_4, Batch1, 3 wells × 3 sites, U2OS cells) from the Cell Painting Gallery and builds a MicroSplit training dataset.

All outputs go to `./cpg0016_demo/` — delete that folder to start fresh.

Run `01_noisemodels.ipynb` next.

In [ ]:
import sys, csv, io
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tifffile
import pandas as pd
import boto3
from botocore import UNSIGNED
from botocore.config import Config

sys.path.insert(0, '../../../src')
from microsplit_reproducibility.workflows.cellpainting import combine_channels, save_dataset_images

DEMO_DIR    = Path('./cpg0016_demo')
IMAGES_DIR  = DEMO_DIR / 'images'
DATASET_DIR = DEMO_DIR / 'dataset'

CHANNELS = ['DNA', 'RNA', 'ER', 'AGP', 'Mito']
CH_IDX   = {'DNA': 5, 'RNA': 3, 'ER': 4, 'AGP': 2, 'Mito': 1}

# cpg0016 source_4, Batch1 — U2OS, standard 5-channel Cell Painting
SOURCE       = 'source_4'
BATCH        = '2021_04_26_Batch1'
PLATE        = 'BR00117035'
PLATE_FOLDER = 'BR00117035__2021-05-02T16_02_51-Measurement1'
WELLS        = ['B02', 'B03', 'C02']
N_SITES      = 3

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def _well_to_rowcol(well: str):
    row = ord(well[0].upper()) - ord('A') + 1
    col = int(well[1:])
    return row, col

s3     = boto3.client('s3', config=Config(signature_version=UNSIGNED))
BUCKET = 'cellpainting-gallery'

downloaded = []
for well in WELLS:
    row, col = _well_to_rowcol(well)
    for site in range(1, N_SITES + 1):
        for ch in CHANNELS:
            out_path = IMAGES_DIR / f'{well}_s{site:02d}_{ch}.tiff'
            if out_path.exists():
                downloaded.append((well, site, ch, out_path))
                continue
            key = (
                f'cpg0016-jump/{SOURCE}/images/{BATCH}/images/'
                f'{PLATE_FOLDER}/Images/'
                f'r{row:02d}c{col:02d}f{site:02d}p01-ch{CH_IDX[ch]}sk1fk1fl1.tiff'
            )
            try:
                buf = io.BytesIO()
                s3.download_fileobj(BUCKET, key, buf)
                buf.seek(0)
                img = tifffile.imread(buf)
                tifffile.imwrite(str(out_path), img)
                downloaded.append((well, site, ch, out_path))
                print(f'  {out_path.name}')
            except Exception as e:
                print(f'  FAILED {well} s{site} {ch}: {e}')

print(f'{len(downloaded)} images ready')

In [ ]:
well, site = 'B02', 1
fig, axes = plt.subplots(1, len(CHANNELS), figsize=(18, 4))
for ax, ch in zip(axes, CHANNELS):
    img = tifffile.imread(str(IMAGES_DIR / f'{well}_s{site:02d}_{ch}.tiff'))
    ax.imshow(img, cmap='gray', vmin=0, vmax=np.percentile(img, 99.5))
    ax.set_title(f'{ch}\n{img.shape}  {img.dtype}')
    ax.axis('off')
fig.suptitle(f'cpg0016 U2OS | {PLATE} | {well} site {site}', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
metadata_rows = []
image_id = 0

for well in WELLS:
    for site in range(1, N_SITES + 1):
        channel_images = {}
        paths_ok = True
        for ch in CHANNELS:
            path = IMAGES_DIR / f'{well}_s{site:02d}_{ch}.tiff'
            if not path.exists():
                paths_ok = False
                break
            channel_images[ch] = tifffile.imread(str(path))
        if not paths_ok:
            continue

        combined = combine_channels(channel_images, CHANNELS)
        save_dataset_images(combined, channel_images, image_id, DATASET_DIR, CHANNELS)
        metadata_rows.append({
            'image_id': image_id, 'source': SOURCE, 'batch': BATCH,
            'plate': PLATE, 'well': well, 'site': site,
            **{f'{ch}_path': f'{ch}/{image_id:06d}.tiff' for ch in CHANNELS},
            'combined_path': f'combined/{image_id:06d}.tiff',
        })
        image_id += 1

with open(DATASET_DIR / 'metadata.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(metadata_rows[0].keys()))
    writer.writeheader()
    writer.writerows(metadata_rows)

print(f'{image_id} images → {DATASET_DIR}')

In [ ]:
meta = pd.read_csv(DATASET_DIR / 'metadata.csv')
display(meta.head())

combined_img = tifffile.imread(str(DATASET_DIR / 'combined' / '000000.tiff'))
fig, axes = plt.subplots(1, len(CHANNELS) + 1, figsize=(22, 4))
axes[0].imshow(combined_img, cmap='viridis', vmin=0, vmax=np.percentile(combined_img, 99.5))
axes[0].set_title('combined')
axes[0].axis('off')
for ax, ch in zip(axes[1:], CHANNELS):
    img = tifffile.imread(str(DATASET_DIR / ch / '000000.tiff'))
    ax.imshow(img, cmap='gray', vmin=0, vmax=np.percentile(img, 99.5))
    ax.set_title(ch)
    ax.axis('off')
plt.tight_layout()
plt.show()